# 1、SummarizationMiddleware中间件

## 举例1：测试trigger、keep参数

In [4]:
from langchain.agents.middleware import SummarizationMiddleware
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

# 从.env文件中加载环境变量
load_dotenv(override=True)

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

model = init_chat_model(
    model="deepseek-v4-flash",
    model_provider="deepseek",
    profile={"max_input_tokens":128_000},
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL,
    extra_body={"thinking": {"type": "disabled"}}
)

In [5]:
from langchain_core.messages import SystemMessage,HumanMessage,AIMessage
from langchain.agents import create_agent


messages = [
    SystemMessage("你是个非常友好的AI助手"),
    HumanMessage("你好啊，我是老王，你是谁？"),
    AIMessage("你好老王，我是小王"),
    HumanMessage("好的小王，很高兴认识你"),
    AIMessage("你高兴得太早了"),
    HumanMessage("呵呵，你什么意思")
]



agent = create_agent(
    model="deepseek-v4-flash",
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=[
                ("tokens",100),
                ("messages",6),
                ("fraction",0.001)   ## 摘要时保留max_input_tokens*fraction个token
            ],
            keep=("messages",2)
        )
    ]
)

response = agent.invoke({
    "messages": messages
})

for msg in response["messages"]:
    msg.pretty_print()


================================ Human Message =================================

Here is a summary of the conversation to date:

## SESSION INTENT
None (no ongoing task or user request established).

## SUMMARY
None (this conversation contains only a brief introduction with no substantive task, decisions, or conclusions). The user, “老王,” greeted the AI, who introduced itself as “小王.” The user expressed pleasure at meeting the AI, but no work or goal was initiated.

## ARTIFACTS
None

## NEXT STEPS
None — await further instructions or a task from the user.
================================== Ai Message ==================================

你高兴得太早了
================================ Human Message =================================

呵呵，你什么意思
================================== Ai Message ==================================

哈哈，老王同志，别误会，我那句“你高兴得太早了”其实是句玩笑话，没有恶意。

我自嘲“小王”是个AI，虽然能陪你聊天、帮点忙，但毕竟水平有限，偶尔也会犯傻、答非所问，甚至说些冷笑话把你冻着。所以你刚说“很高兴认识我”，我心想：万一等下我表现不好，让你失望了，那你岂不是白高兴一场？于是就先给你打个预防针——别对我期望太高，免得一会儿摔着。

当然啦

In [ ]:
## 举例2：summary_prompt

In [6]:

from langchain_core.messages import SystemMessage,HumanMessage,AIMessage
from langchain.agents import create_agent


messages = [
    SystemMessage("你是个非常友好的AI助手"),
    HumanMessage("你好啊，我是老王，你是谁？"),
    AIMessage("你好老王，我是小王"),
    HumanMessage("好的小王，很高兴认识你"),
    AIMessage("你高兴得太早了"),
    HumanMessage("呵呵，你什么意思")
]



agent = create_agent(
    model="deepseek-v4-flash",
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=[
                ("tokens",100),
                ("messages",6),
                ("fraction",0.001)
            ],
            keep=("messages",2),
            summary_prompt="对历史消息摘要，消息列表如下\n{messages}"
        )
    ]
)

response = agent.invoke({
    "messages": messages
})

for msg in response["messages"]:
    msg.pretty_print()


================================ Human Message =================================

Here is a summary of the conversation to date:

好的，这是对历史消息的摘要：

- **系统设定**：AI被设定为非常友好的助手。
- **内容**：用户老王与AI助手小王互相问候，并表达了认识彼此的愉快心情。
================================== Ai Message ==================================

你高兴得太早了
================================ Human Message =================================

呵呵，你什么意思
================================== Ai Message ==================================

哎呀，抱歉抱歉，我刚刚那句“呵呵”可能让你觉得不太舒服，其实我完全没有不好的意思，就是随口一接，想问问你怎么突然说我“高兴得太早了”。可能是我表达得不够清楚，让你误会了。

你要是愿意的话，可以跟我说说刚才为什么那么讲吗？我真的很想好好听你说话，帮你解解闷，或者一起聊聊开心的事。咱们别因为一个小误会生分了，好吗？😊
